In [5]:
!pip install langchain chromadb openai tiktoken pypdf langchain_openai langchain-community

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import Chroma

In [11]:
from langchain_core.documents import Document

# Create LangChain documents for IPL players

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )

In [12]:
docs = [doc1 , doc2 , doc3 , doc4 , doc5]

use this model -> ollama pull nomic-embed-text


In [18]:
vector_store = Chroma(
    embedding_function=OllamaEmbeddings(model="nomic-embed-text"),
    persist_directory="my_chroma_db",
    collection_name="sample"
)


In [19]:
vector_store.add_documents(docs)

['50f3d7e2-349a-4513-a374-f741f451db78',
 'e9770db8-69a5-49f8-8507-34b2255b6310',
 '07dfe49d-cc3d-41f4-9bea-a248b8d88eca',
 'b47d0631-6bb6-4310-aa9b-4bebeeef9b8b',
 'ea8fa384-ad74-4a7b-a2fd-d657d64057a8']

In [21]:
# view documents 
vector_store.get(include=['embeddings' , "metadatas" , "documents"])

{'ids': ['50f3d7e2-349a-4513-a374-f741f451db78',
  'e9770db8-69a5-49f8-8507-34b2255b6310',
  '07dfe49d-cc3d-41f4-9bea-a248b8d88eca',
  'b47d0631-6bb6-4310-aa9b-4bebeeef9b8b',
  'ea8fa384-ad74-4a7b-a2fd-d657d64057a8'],
 'embeddings': array([[-0.024059  ,  0.04453161, -0.18949507, ..., -0.01605532,
         -0.05859691,  0.0025295 ],
        [-0.03152767,  0.07583491, -0.16415194, ..., -0.0656625 ,
         -0.02191824, -0.01390644],
        [ 0.02280271,  0.0657957 , -0.18388067, ..., -0.021293  ,
         -0.04229931,  0.00040438],
        [ 0.01442985,  0.03144638, -0.18336748, ..., -0.02951371,
         -0.03739943,  0.00836507],
        [-0.03468144,  0.03123768, -0.16423446, ..., -0.0237606 ,
         -0.05102901,  0.03770281]], shape=(5, 768)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the mo

In [23]:
#search documnet 
vector_store.similarity_search(
    query = "Who is bowler ? ",
    k=1
)

[Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.')]

In [24]:
# search with similarity score
vector_store.similarity_search_with_score(
    query='Who among these are a bowler?',
    k=2
)

[(Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.6737993359565735),
 (Document(metadata={'team': 'Royal Challengers Bangalore'}, page_content='Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.'),
  0.8671941757202148)]

In [25]:
# meta-data filtering
vector_store.similarity_search_with_score(
    query="",
    filter={"team": "Chennai Super Kings"}
)

[(Document(metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  0.8770320415496826),
 (Document(metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  0.9054960608482361)]

In [26]:
# update documents
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

vector_store.update_document(document_id='09a39dc6-3ba6-4ea7-927e-fdda591da5e4', document=updated_doc1)

In [27]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['50f3d7e2-349a-4513-a374-f741f451db78',
  'e9770db8-69a5-49f8-8507-34b2255b6310',
  '07dfe49d-cc3d-41f4-9bea-a248b8d88eca',
  'b47d0631-6bb6-4310-aa9b-4bebeeef9b8b',
  'ea8fa384-ad74-4a7b-a2fd-d657d64057a8'],
 'embeddings': array([[-0.024059  ,  0.04453161, -0.18949507, ..., -0.01605532,
         -0.05859691,  0.0025295 ],
        [-0.03152767,  0.07583491, -0.16415194, ..., -0.0656625 ,
         -0.02191824, -0.01390644],
        [ 0.02280271,  0.0657957 , -0.18388067, ..., -0.021293  ,
         -0.04229931,  0.00040438],
        [ 0.01442985,  0.03144638, -0.18336748, ..., -0.02951371,
         -0.03739943,  0.00836507],
        [-0.03468144,  0.03123768, -0.16423446, ..., -0.0237606 ,
         -0.05102901,  0.03770281]], shape=(5, 768)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the mo

In [28]:
# delete documents 
vector_store.delete(ids=["09a39dc6-3ba6-4ea7-927e-fdda591da5e4"])

In [29]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['50f3d7e2-349a-4513-a374-f741f451db78',
  'e9770db8-69a5-49f8-8507-34b2255b6310',
  '07dfe49d-cc3d-41f4-9bea-a248b8d88eca',
  'b47d0631-6bb6-4310-aa9b-4bebeeef9b8b',
  'ea8fa384-ad74-4a7b-a2fd-d657d64057a8'],
 'embeddings': array([[-0.024059  ,  0.04453161, -0.18949507, ..., -0.01605532,
         -0.05859691,  0.0025295 ],
        [-0.03152767,  0.07583491, -0.16415194, ..., -0.0656625 ,
         -0.02191824, -0.01390644],
        [ 0.02280271,  0.0657957 , -0.18388067, ..., -0.021293  ,
         -0.04229931,  0.00040438],
        [ 0.01442985,  0.03144638, -0.18336748, ..., -0.02951371,
         -0.03739943,  0.00836507],
        [-0.03468144,  0.03123768, -0.16423446, ..., -0.0237606 ,
         -0.05102901,  0.03770281]], shape=(5, 768)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the mo